## Full Dataset Preprocessing

### Setup   

In [1]:
# libraries
import matplotlib.pyplot as plt ## for basic plotting
import matplotlib as mpl ## for setting default parameters
import pandas as pd ## always
import os ## for handling file paths 1
from pathlib import Path ## for handling file paths 2
import numpy as np ## for numerical operations
import seaborn as sns ## for more advanced plotting

In [2]:
# paths
base_dir = Path("/Users/hannahmaihojgaard/Documents/GitHub/datascience2026")
data_path = Path(base_dir / "data")

### Load in data

In [3]:
# load cleaned taxi datasets
#taxi_df_1 = pd.read_parquet(
#    data_path / "taxi_cleaned_part1.parquet"
#)

taxi_df_2 = pd.read_parquet(
    data_path / "taxi_cleaned_part2.parquet"
)

KeyboardInterrupt: 

In [40]:
# load cleaned oil dataset
oil_df = pd.read_parquet(
    data_path / "oil_cleaned_daily.parquet"
)

In [ ]:
#print(taxi_df_1.shape)
print(taxi_df_2.shape)
print(oil_df.shape)

(86173866, 22)
(1389, 2)


: 

### Combine Oil and taxi data

In [ ]:
# create date column
taxi_df_2["date"] = taxi_df_2["tpep_pickup_datetime"].dt.floor("D")

# merge oil prices into part 2
taxi_df_2 = taxi_df_2.merge(
    oil_df,
    on="date",
    how="left"
)

In [ ]:
# check missing oil prices
taxi_df_2["oil_price"].isna().sum()

23304136

In [ ]:
# create complete daily oil dataset
oil_daily = pd.DataFrame({
    "date": pd.date_range(
        start=taxi_df_2["date"].min(),
        end=oil_df["date"].max(),
        freq="D"
    )
})

oil_daily = oil_daily.merge(
    oil_df,
    on="date",
    how="left"
)

# fill missing oil prices
oil_daily["oil_price"] = oil_daily["oil_price"].bfill().ffill()

# check
oil_daily["oil_price"].isna().sum()

0

In [ ]:
# remove old oil_price column
taxi_df_2 = taxi_df_2.drop(columns=["oil_price"])

# map oil prices by date
oil_map = oil_daily.set_index("date")["oil_price"]

taxi_df_2["oil_price"] = taxi_df_2["date"].map(oil_map)

# check missing values again
taxi_df_2["oil_price"].isna().sum()

0

In [ ]:
taxi_df_2.loc[
    taxi_df_2["oil_price"].isna(),
    "date"
].value_counts().sort_index()

Series([], Name: count, dtype: int64)

In [ ]:
taxi_df_2.head(100)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,source_file,cbd_congestion_fee,trip_duration,date,oil_price
0,1,2023-07-01 00:29:59,2023-07-01 00:40:15,1.0,1.80,1.0,N,140,263,1,...,0.0,1.0,22.20,2.5,0.0,2023-07,0.0,10.266667,2023-07-01,74.52
1,2,2023-07-01 00:03:25,2023-07-01 00:23:44,1.0,2.31,1.0,N,163,163,2,...,0.0,1.0,24.10,2.5,0.0,2023-07,0.0,20.316667,2023-07-01,74.52
2,2,2023-07-01 00:38:29,2023-07-01 00:48:53,1.0,2.36,1.0,N,142,262,1,...,0.0,1.0,22.20,2.5,0.0,2023-07,0.0,10.400000,2023-07-01,74.52
3,2,2023-07-01 00:14:16,2023-07-01 00:29:13,1.0,4.36,1.0,N,68,24,1,...,0.0,1.0,29.76,2.5,0.0,2023-07,0.0,14.950000,2023-07-01,74.52
4,1,2023-07-01 00:11:15,2023-07-01 00:20:47,0.0,1.60,1.0,N,161,107,1,...,0.0,1.0,19.65,2.5,0.0,2023-07,0.0,9.533333,2023-07-01,74.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2,2023-07-01 00:20:16,2023-07-01 00:30:49,1.0,2.33,1.0,N,239,262,2,...,0.0,1.0,18.50,2.5,0.0,2023-07,0.0,10.550000,2023-07-01,74.52
96,2,2023-07-01 00:35:43,2023-07-01 00:48:41,1.0,2.94,1.0,N,263,48,1,...,0.0,1.0,24.72,2.5,0.0,2023-07,0.0,12.966667,2023-07-01,74.52
97,2,2023-07-01 00:06:28,2023-07-01 00:20:49,1.0,2.51,1.0,N,249,233,1,...,0.0,1.0,20.80,2.5,0.0,2023-07,0.0,14.350000,2023-07-01,74.52
98,2,2023-07-01 00:22:18,2023-07-01 00:39:06,1.0,3.18,1.0,N,79,230,1,...,0.0,1.0,23.40,2.5,0.0,2023-07,0.0,16.800000,2023-07-01,74.52


In [ ]:
taxi_df_2['oil_price'].describe()

count    8.617387e+07
mean     7.650567e+01
std      8.595888e+00
min      5.993000e+01
25%      6.933000e+01
50%      7.596000e+01
75%      8.325000e+01
max      9.710000e+01
Name: oil_price, dtype: float64

In [ ]:
taxi_df_2.to_parquet(
    data_path / "taxi_cleaned_part2_with_oil.parquet",
    index=False
)

# Creating the Test data
Random samples from 24/25

In [4]:
# load second cleaned dataset
taxi_df_2 = pd.read_parquet(
    data_path / "taxi_cleaned_part1_with_oil.parquet"
)

In [5]:
print(taxi_df_2["tpep_pickup_datetime"].min())
print(taxi_df_2["tpep_pickup_datetime"].max())

2023-06-30 14:23:27
2025-12-31 23:59:54


In [6]:
taxi_2425 = taxi_df_2[
    taxi_df_2["tpep_pickup_datetime"].dt.year.isin([2024, 2025])
]

In [7]:
taxi_2425.shape

(68584518, 24)

In [8]:
print(taxi_2425["tpep_pickup_datetime"].min())
print(taxi_2425["tpep_pickup_datetime"].max())

2024-01-01 00:00:00
2025-12-31 23:59:54


In [9]:
# sample 3 million rows for training
train_df = taxi_2425.sample(
    n=3_000_000,
    random_state=42
)

In [10]:
train_df.shape

(3000000, 24)

In [11]:
train_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,source_file,cbd_congestion_fee,trip_duration,date,oil_price
26833452,2,2024-04-07 13:26:07,2024-04-07 13:41:27,2.0,2.91,1.0,N,164,141,2,...,0.00,1.0,20.30,2.5,0.0,2024-04,0.00,15.333333,2024-04-07,91.73
53016255,2,2025-01-24 20:39:17,2025-01-24 20:50:24,2.0,1.89,1.0,N,114,209,1,...,0.00,1.0,22.26,2.5,0.0,2025-01,0.75,11.116667,2025-01-24,78.71
79376599,1,2025-10-24 12:15:20,2025-10-24 12:34:03,1.0,1.50,1.0,N,230,264,1,...,0.00,1.0,25.25,2.5,0.0,2025-10,0.75,18.716667,2025-10-24,65.80
67877177,2,2025-06-22 11:19:53,2025-06-22 11:23:35,3.0,1.79,5.0,N,129,138,1,...,6.94,1.0,87.36,0.0,0.0,2025-06,0.00,3.700000,2025-06-22,74.34
28418146,2,2024-04-23 11:37:09,2024-04-23 11:56:47,1.0,1.57,1.0,N,161,140,1,...,0.00,1.0,26.04,2.5,0.0,2024-04,0.00,19.633333,2024-04-23,88.29


In [12]:
train_df.to_parquet(
    data_path / "taxi_train_2024_2025.parquet",
    index=False
)